# **Visão geral do notebook**

Este notebook implementa um pipeline completo para geração de dados pessoais sintéticos com foco em reprodutibilidade, consistência estrutural e governança. O fluxo é dividido em células para facilitar execução incremental no Google Colab:

1. Instalação de dependências

2. Configuração do ambiente, seed e geradores determinísticos

3. Construção do dataset base e componentes da GAN

4. Treinamento + geração com métricas + pós-processamento + exportação

5. Validações finais e inspeção do resultado

## **Célula 1 — Instalação de dependências**

Nesta célula são instaladas as bibliotecas necessárias para execução do pipeline no Colab:

- tensorflow: implementação do modelo GAN (Keras)

- faker: geração de nomes brasileiros e perfis coerentes por gênero

- pandas/numpy: manipulação, transformação e exportação dos dados

- scikit-learn: normalização (MinMaxScaler) e utilitários

- openpyxl: suporte à exportação em .xlsx

In [ ]:
%pip install -q tensorflow faker pandas numpy scikit-learn openpyxl

## **Célula 2 — Ambiente, reprodutibilidade e geradores determinísticos**

Esta célula prepara o ambiente e define funções determinísticas para gerar atributos e identificadores com formato válido.

Esses dados são fictícios e usados apenas em testes/homologação, sem relação com titulares reais.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam
from datetime import datetime, timedelta
from faker import Faker

# =========================
# Reprodutibilidade (seed)
# =========================
SEED = 41
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

fake = Faker("pt_BR")

# =========================
# Geradores determinísticos
# =========================
def gerar_data_nascimento_por_idade(idade: int) -> str:
    hoje = datetime.now()
    dias = idade * 365 + random.randint(0, 364)
    data = hoje - timedelta(days=dias)
    return data.strftime("%d/%m/%Y")

def gerar_cpf() -> str:
    def calc_digito(digs):
        s = sum(int(d) * w for d, w in zip(digs, range(len(digs) + 1, 1, -1)))
        r = 11 - (s % 11)
        return "0" if r > 9 else str(r)

    n = [str(random.randint(0, 9)) for _ in range(9)]
    n.append(calc_digito(n))
    n.append(calc_digito(n))
    return f"{''.join(n[:3])}.{''.join(n[3:6])}.{''.join(n[6:9])}-{''.join(n[9:])}"

def gerar_cnh() -> str:
    n = [random.randint(0, 9) for _ in range(9)]
    soma = sum((9 - i) * n[i] for i in range(9))
    d1 = soma % 11
    d1 = 0 if d1 >= 10 else d1

    soma = sum((i + 1) * n[i] for i in range(9))
    d2 = soma % 11
    d2 = 0 if d2 >= 10 else d2

    return "".join(map(str, n)) + str(d1) + str(d2)

def gerar_rg() -> str:
    n = [str(random.randint(0, 9)) for _ in range(8)]
    return f"{''.join(n[:2])}.{''.join(n[2:5])}.{''.join(n[5:8])}-{random.randint(0,9)}"

def gerar_titulo_eleitor() -> str:
    def calc_dv(num, uf):
        d1 = sum(int(num[i]) * (9 - i) for i in range(8)) % 11
        d1 = 0 if d1 == 10 else d1
        d2 = sum(int(num[i]) * (8 - i) for i in range(8)) + d1 * 9 + int(uf) * 10
        d2 = d2 % 11
        d2 = 0 if d2 == 10 else d2
        return str(d1) + str(d2)

    num = "".join(str(random.randint(0, 9)) for _ in range(8))
    uf = f"{random.randint(1, 28):02d}"
    dv = calc_dv(num, uf)
    return f"{num[:4]} {num[4:]} {uf} {dv}"

def gerar_telefone() -> str:
    ddds = [11,12,13,14,15,16,17,18,19,21,22,24,27,28,31,32,33,34,35,37,38,
            41,42,43,44,45,46,47,48,49,51,53,54,55,61,62,63,64,65,66,67,68,
            69,71,73,74,75,77,79,81,82,83,84,85,86,87,88,89,91,92,93,94,
            95,96,97,98,99]
    ddd = random.choice(ddds)
    prefixo = random.randint(90000, 99999)
    sufixo = random.randint(1000, 9999)
    return f"({ddd}) {prefixo}-{sufixo}"

def gerar_renda() -> float:
    # Renda mensal entre R$ 1.200 e R$ 25.000
    return round(random.uniform(1200, 25000), 2)

## **Células 3 e 4 — Base de calibração, pré-processamento e arquitetura da GAN**

Aqui é criada uma base “de calibração” (dados sintéticos simples) e a estrutura da GAN.

**Base de calibração**: Essa base alimenta o treinamento da GAN para aprender distribuições e dependências.



**Pré-processamento (DataPreprocessor)**: Como a GAN trabalha melhor em valores numéricos normalizados, cada coluna é escalada para [0, 1] com MinMaxScaler na volta, o inverse_transform reconstrói valores no domínio original.

**GAN (Generator + Discriminator)**

1. Generator: recebe ruído (latent_dim) e gera vetores tabulares (0..1)

2. Discriminator: tenta distinguir amostras reais (calibração) vs sintéticas

Treinamento ocorre em regime adversarial, alternando atualização de D e G

In [ ]:
def gerar_pessoa_base():
    idade = random.randint(18, 65)
    sexo = random.choice([0, 1])   # 0=feminino, 1=masculino (para treino)
    renda = gerar_renda()
    return {"Idade": idade, "Sexo": sexo, "Renda": renda}

def gerar_dataset_real(n: int) -> pd.DataFrame:
    return pd.DataFrame([gerar_pessoa_base() for _ in range(n)])

class DataPreprocessor:
    def __init__(self):
        self.scalers = {}

    def fit_transform(self, df: pd.DataFrame) -> np.ndarray:
        processed = pd.DataFrame()

        for col in df.columns:
            scaler = MinMaxScaler()
            processed[col] = scaler.fit_transform(df[[col]])
            self.scalers[col] = scaler

        return processed.values

    def inverse_transform(self, data: np.ndarray) -> pd.DataFrame:
        cols = list(self.scalers.keys())
        df = pd.DataFrame(data, columns=cols)
        for col in cols:
            df[col] = self.scalers[col].inverse_transform(df[[col]])
        return df

def build_generator(latent_dim: int, output_dim: int) -> Sequential:
    return Sequential([
        layers.Dense(128, activation="relu", input_dim=latent_dim),
        layers.Dense(256, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(output_dim, activation="sigmoid")
    ])

def build_discriminator(input_dim: int) -> Sequential:
    return Sequential([
        layers.Dense(256, activation="relu", input_dim=input_dim),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

def build_gan(generator: Sequential, discriminator: Sequential, latent_dim: int) -> Sequential:
    discriminator.trainable = False
    gan = Sequential([generator, discriminator])
    gan.compile(loss="binary_crossentropy", optimizer=Adam(0.0001, 0.5))
    discriminator.trainable = True
    return gan

def train_gan(generator: Sequential,
              discriminator: Sequential,
              gan: Sequential,
              data: np.ndarray,
              latent_dim: int,
              epochs: int = 100,
              batch_size: int = 64):

    real_labels = np.ones((batch_size, 1))
    fake_labels = np.zeros((batch_size, 1))

    for epoch in range(epochs):
        # ---- Treina Discriminador ----
        idx = np.random.randint(0, data.shape[0], batch_size)
        real_samples = data[idx]

        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_samples = generator.predict(noise, verbose=0)

        d_loss_real = discriminator.train_on_batch(real_samples, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # ---- Treina Gerador (via GAN com D congelado) ----
        discriminator.trainable = False
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = gan.train_on_batch(noise, real_labels)
        discriminator.trainable = True

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | D loss: {float(d_loss[0]):.4f} | G loss: {float(g_loss):.4f}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

class DataPreprocessor:
    def __init__(self):
        self.scalers = {}
        self.columns = None

    def fit_transform(self, df: pd.DataFrame) -> np.ndarray:
        self.columns = list(df.columns)

        # garante index (evita erro de frame sem index)
        processed = pd.DataFrame(index=df.index)

        for col in self.columns:
            scaler = MinMaxScaler()
            processed[col] = scaler.fit_transform(df[[col]]).ravel()  # <- 1D
            self.scalers[col] = scaler

        return processed.to_numpy(dtype=np.float32)

    def inverse_transform(self, data: np.ndarray) -> pd.DataFrame:
        df = pd.DataFrame(data, columns=self.columns)

        for col in self.columns:
            df[col] = self.scalers[col].inverse_transform(df[[col]]).ravel()  # <- 1D

        return df

## **Célula 5 e 6 — Execução principal (treino + geração + métricas + pós-processamento) e amostra gerada**

Estas células executam o pipeline completo.

In [ ]:
import time
import json
from collections import Counter

def gerar_sinteticos_com_metricas(
    generator,
    discriminator,
    preprocessor,
    latent_dim: int,
    n_target: int = 1000,
    batch_gen: int = 2048,
    score_threshold: float = 0.50,
    max_batches: int = 200
):
    """
    Gera amostras até atingir n_target aceitas.
    Critérios de aceitação:
      (1) Regras semânticas simples (idade/renda/sexo dentro do domínio)
      (2) Escore do discriminador >= score_threshold
    Retorna: df_aceitos, relatorio(dict)
    """
    t0 = time.perf_counter()

    total_candidatos = 0
    total_aceitos = 0

    rejeicoes = Counter()  # motivos

    aceitos_scaled = []    # amostras aceitas no espaço escalado (sigmoid output)
    aceitos_orig = []      # amostras aceitas no espaço original (inverse_transform)

    for _ in range(max_batches):
        # 1) gera batch no espaço escalado (0..1)
        noise = np.random.normal(0, 1, (batch_gen, latent_dim))
        gen_scaled = generator.predict(noise, verbose=0)  # shape: (batch_gen, n_features)
        total_candidatos += batch_gen

        # 2) escore do discriminador (quanto mais alto, mais "real")
        scores = discriminator.predict(gen_scaled, verbose=0).reshape(-1)  # prob real

        # 3) converte para original e aplica regras semânticas
        df_orig = preprocessor.inverse_transform(gen_scaled)

        # Regras simples (ajuste se quiser)
        idade = df_orig["Idade"]
        sexo = df_orig["Sexo"]
        renda = df_orig["Renda"]

        # máscara de validade por domínio (sem "clip" — aqui rejeita mesmo)
        mask_dom = (
            idade.between(18, 65)
            & renda.between(1200, 25000)
            & sexo.between(0, 1)
        )

        # máscara do discriminador
        mask_disc = scores >= score_threshold

        # aceitação final
        mask_ok = mask_dom & mask_disc
        n_ok = int(mask_ok.sum())

        # contabiliza rejeições
        rejeicoes["rejeitado_total"] += int((~mask_ok).sum())
        rejeicoes["rejeitado_disc"] += int((mask_dom & ~mask_disc).sum())
        rejeicoes["rejeitado_dom"] += int((~mask_dom).sum())

        # salva aceitos
        if n_ok > 0:
            aceitos_scaled.append(gen_scaled[mask_ok])
            aceitos_orig.append(df_orig.loc[mask_ok])

            total_aceitos += n_ok
            if total_aceitos >= n_target:
                break

    if total_aceitos == 0:
        raise RuntimeError("Nenhuma amostra foi aceita. Reduza o score_threshold ou aumente max_batches.")

    # concatena e corta para n_target
    X_scaled = np.vstack(aceitos_scaled)[:n_target]
    df_final = pd.concat(aceitos_orig, ignore_index=True).iloc[:n_target].copy()

    t1 = time.perf_counter()
    tempo = t1 - t0

    # métricas resumidas
    relatorio = {
        "n_target": int(n_target),
        "score_threshold": float(score_threshold),
        "batch_gen": int(batch_gen),
        "max_batches": int(max_batches),
        "total_candidatos": int(total_candidatos),
        "total_aceitos": int(n_target),
        "total_rejeitados": int(total_candidatos - n_target),
        "taxa_aceitacao": float(n_target / total_candidatos),
        "tempo_geracao_seg": float(tempo),
        "throughput_aceitos_por_seg": float(n_target / tempo),
        "rejeicoes": dict(rejeicoes),
        # resumo univariado (antes de qualquer pós-processamento)
        "resumo_univariado": {
            "idade_media": float(df_final["Idade"].mean()),
            "idade_dp": float(df_final["Idade"].std()),
            "renda_media": float(df_final["Renda"].mean()),
            "renda_mediana": float(df_final["Renda"].median()),
            "renda_dp": float(df_final["Renda"].std()),
            "sexo_prop_1": float((df_final["Sexo"].round().clip(0, 1) == 1).mean()),
        }
    }

    return df_final, X_scaled, relatorio

In [ ]:
def main():
    import json

    print("Gerando base de calibração (dados para treino)...")
    real_data = gerar_dataset_real(20000)

    print("Pré-processando...")
    preprocessor = DataPreprocessor()
    processed_data = preprocessor.fit_transform(real_data)

    print("Construindo e treinando a GAN...")
    latent_dim = 16
    output_dim = processed_data.shape[1]

    generator = build_generator(latent_dim, output_dim)
    discriminator = build_discriminator(output_dim)
    discriminator.compile(
        loss="binary_crossentropy",
        optimizer=Adam(0.0001, 0.5),
        metrics=["accuracy"]
    )

    gan = build_gan(generator, discriminator, latent_dim)

    train_gan(
        generator,
        discriminator,
        gan,
        processed_data,
        latent_dim,
        epochs=100,
        batch_size=64
    )

    print("Gerando dados sintéticos (GAN) com métricas de aceitação/rejeição...")
    synthetic, synthetic_scaled, relatorio = gerar_sinteticos_com_metricas(
        generator=generator,
        discriminator=discriminator,
        preprocessor=preprocessor,
        latent_dim=latent_dim,
        n_target=1000,
        batch_gen=2048,
        score_threshold=0.50,
        max_batches=200
    )

    print("Resumo da geração:")
    print(json.dumps(relatorio, indent=2, ensure_ascii=False))

    # ========= Pós-processamento coerente =========

    # Arredonda e limita idade (para DOB e coerência)
    synthetic["Idade"] = synthetic["Idade"].round().astype(int).clip(18, 65)

    # Converter Sexo para Gênero
    synthetic["Gênero"] = synthetic["Sexo"].round().astype(int).clip(0, 1)
    synthetic["Gênero"] = synthetic["Gênero"].map({0: "Feminino", 1: "Masculino"})

    # Remover coluna original Sexo
    synthetic = synthetic.drop(columns=["Sexo"])

    # Data nascimento coerente com idade
    synthetic["Data_Nascimento"] = synthetic["Idade"].apply(gerar_data_nascimento_por_idade)

    # Nome coerente com gênero
    def gerar_nome_por_genero(genero_label: str) -> str:
        if genero_label == "Feminino":
            return fake.first_name_female() + " " + fake.last_name()
        return fake.first_name_male() + " " + fake.last_name()

    synthetic["Nome"] = synthetic["Gênero"].apply(gerar_nome_por_genero)

    # Documentos e telefone
    synthetic["CPF"] = [gerar_cpf() for _ in range(len(synthetic))]
    synthetic["CNH"] = [gerar_cnh() for _ in range(len(synthetic))]
    synthetic["RG"] = [gerar_rg() for _ in range(len(synthetic))]
    synthetic["Titulo_Eleitor"] = [gerar_titulo_eleitor() for _ in range(len(synthetic))]
    synthetic["Telefone"] = [gerar_telefone() for _ in range(len(synthetic))]

    # Renda com duas casas decimais
    synthetic["Renda"] = synthetic["Renda"].astype(float).round(2)

    relatorio["colisoes_cpf"] = int(synthetic["CPF"].duplicated().sum())

    # Remover idade do arquivo final (se não quiser expor)
    synthetic = synthetic.drop(columns=["Idade"])

    # Ordenar colunas finais
    cols = [
        "Nome",
        "Gênero",
        "Data_Nascimento",
        "CPF",
        "CNH",
        "RG",
        "Titulo_Eleitor",
        "Telefone",
        "Renda"
    ]
    synthetic = synthetic[cols]

    synthetic.to_excel("dados_sinteticos_realistas.xlsx", index=False)

    # Salvar relatório (bom para o artigo)
    with open("relatorio_execucao.json", "w", encoding="utf-8") as f:
        json.dump(relatorio, f, indent=2, ensure_ascii=False)

    print("OK! Arquivo salvo: dados_sinteticos_realistas.xlsx")
    print("OK! Relatório salvo: relatorio_execucao.json")


if __name__ == "__main__":
    main()

Gerando base de calibração (dados para treino)...
Pré-processando...
Construindo e treinando a GAN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100 | D loss: 0.7047 | G loss: 0.7094
Epoch 10/100 | D loss: 0.6907 | G loss: 0.7100
Epoch 20/100 | D loss: 0.6860 | G loss: 0.7096
Epoch 30/100 | D loss: 0.6825 | G loss: 0.7068
Epoch 40/100 | D loss: 0.6799 | G loss: 0.7039
Epoch 50/100 | D loss: 0.6784 | G loss: 0.6991
Epoch 60/100 | D loss: 0.6771 | G loss: 0.6950
Epoch 70/100 | D loss: 0.6728 | G loss: 0.6963
Epoch 80/100 | D loss: 0.6664 | G loss: 0.7007
Epoch 90/100 | D loss: 0.6599 | G loss: 0.7043
Epoch 100/100 | D loss: 0.6574 | G loss: 0.6991
Gerando dados sintéticos (GAN) com métricas de aceitação/rejeição...
Resumo da geração:
{
  "n_target": 1000,
  "score_threshold": 0.5,
  "batch_gen": 2048,
  "max_batches": 200,
  "total_candidatos": 2048,
  "total_aceitos": 1000,
  "total_rejeitados": 1048,
  "taxa_aceitacao": 0.48828125,
  "tempo_geracao_seg": 0.9458453060001375,
  "throughput_aceitos_por_seg": 1057.2553393840649,
  "rejeicoes": {
    "rejeitado_total": 0,
    "rejeitado_disc": 0,
    "rejeitado_dom": 0
  },


## **Célula 7 — Visualização dos dados gerados**

In [ ]:
data_visualization = pd.read_excel("dados_sinteticos_realistas.xlsx")
data_visualization.head(20)

,Nome,Gênero,Data_Nascimento,CPF,CNH,RG,Titulo_Eleitor,Telefone,Renda
0,Lavínia Siqueira,Feminino,18/08/1997,258.075.909-37,44381457991,41.255.166-9,9455 7157 10 49,(67) 94714-5224,2119.66
1,Bárbara Cassiano,Feminino,16/09/1993,264.710.999-08,37590716501,98.629.906-6,2551 6247 11 93,(47) 99737-7547,2018.98
2,Alexia Albuquerque,Feminino,29/05/1993,848.601.051-92,20615506026,73.432.877-4,5325 6225 08 24,(61) 91761-2481,4344.23
3,Noah Leão,Masculino,04/10/1994,356.484.015-07,41985009559,50.216.725-6,1559 4670 03 50,(73) 95737-3014,2870.21
4,Mathias Sampaio,Masculino,21/02/1995,927.284.443-66,8809115615,26.058.917-1,2383 5241 23 96,(11) 95704-5222,2302.21
5,Sophia Câmara,Feminino,24/11/1996,486.036.103-20,4003331701,58.510.512-3,7791 3507 05 65,(47) 99709-6834,1808.11
6,Isis Nunes,Feminino,10/10/1995,839.091.824-29,99885947142,06.992.522-8,4155 7299 26 08,(79) 91239-7687,3822.67
7,Maria Laura Sampaio,Feminino,29/03/1995,052.636.183-28,79135617060,03.628.630-3,5652 2490 12 64,(34) 92313-1908,2761.02
8,Sabrina Marques,Feminino,29/10/1995,081.396.507-14,43687692867,18.080.909-3,4218 8861 25 12,(54) 91222-3300,2648.06
9,Murilo Vasconcelos,Masculino,09/01/1997,160.667.364-50,46256198301,09.703.901-6,3857 0087 20 80,(64) 92962-9854,2119.28


## **Célula 8 e 9 — Validações finais e inspeção**

Aqui são feitas checagens objetivas para embasar resultados no artigo.

In [ ]:
import re
import pandas as pd

cpf_re = re.compile(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$")
rg_re = re.compile(r"^\d{2}\.\d{3}\.\d{3}-\d$")
tel_re = re.compile(r"^\(\d{2}\) \d{5}-\d{4}$")

def avaliar_regras_final(df: pd.DataFrame) -> dict:
    n = len(df)

    # Regras simples e objetivas (ajuste se quiser)
    v = {}
    v["cpf_formato_invalido"] = int((~df["CPF"].astype(str).str.match(cpf_re)).sum())
    v["rg_formato_invalido"]  = int((~df["RG"].astype(str).str.match(rg_re)).sum())
    v["tel_formato_invalido"] = int((~df["Telefone"].astype(str).str.match(tel_re)).sum())

    # Duplicidade (colisão) — importante pra “controle de risco”
    v["cpf_duplicado"] = int(df["CPF"].duplicated().sum())
    v["cnh_duplicada"] = int(df["CNH"].duplicated().sum())
    v["rg_duplicado"]  = int(df["RG"].duplicated().sum())
    v["titulo_duplicado"] = int(df["Titulo_Eleitor"].duplicated().sum())
    v["telefone_duplicado"] = int(df["Telefone"].duplicated().sum())

    # Se você tiver Idade ainda nesse ponto:
    if "Idade" in df.columns:
        v["idade_fora_faixa"] = int((~df["Idade"].between(18, 65)).sum())

    # Taxa global de violação (linha com qualquer problema)
    viol_por_linha = pd.Series(False, index=df.index)
    for k, cnt in v.items():
        if k.endswith("_invalido") or k.endswith("_fora_faixa"):
            # para inválidos/fora_faixa dá pra marcar por linha
            pass

    # Para uma taxa simples: considera violação se existir qualquer contagem > 0
    total_erros = sum(v.values())
    taxa_erros_por_registro = total_erros / n

    return {
        "n_registros": n,
        "contagens": v,
        "total_erros": int(total_erros),
        "taxa_erros_por_registro": float(taxa_erros_por_registro),
        "taxa_conformidade_aproximada": float(1 - min(1, taxa_erros_por_registro))
    }

synthetic = pd.read_excel("dados_sinteticos_realistas.xlsx")
metricas_final = avaliar_regras_final(synthetic.copy())
metricas_final

{'n_registros': 1000,
 'contagens': {'cpf_formato_invalido': 0,
  'rg_formato_invalido': 0,
  'tel_formato_invalido': 0,
  'cpf_duplicado': 0,
  'cnh_duplicada': 0,
  'rg_duplicado': 0,
  'titulo_duplicado': 0,
  'telefone_duplicado': 0},
 'total_erros': 0,
 'taxa_erros_por_registro': 0.0,
 'taxa_conformidade_aproximada': 1.0}

In [ ]:
def avaliar_regras_bruto(df: pd.DataFrame) -> dict:
    n = len(df)
    viol = {}

    # Check for 'Idade' column before accessing
    if "Idade" in df.columns:
        idade = df["Idade"]
        viol["idade_menor_18"] = int((idade < 18).sum())
        viol["idade_maior_65"] = int((idade > 65).sum())
        # Renda alta para menor só faz sentido se 'Idade' e 'Renda' estão presentes
        if "Renda" in df.columns:
            viol["renda_alta_para_menor"] = int(((idade < 18) & (df["Renda"] > 8000)).sum())
        else:
            viol["renda_alta_para_menor"] = 0
    else:
        viol["idade_menor_18"] = 0
        viol["idade_maior_65"] = 0
        viol["renda_alta_para_menor"] = 0

    # Check for 'Renda' column before accessing
    if "Renda" in df.columns:
        renda = df["Renda"]
        viol["renda_fora_faixa"] = int((~renda.between(1200, 25000)).sum())
    else:
        viol["renda_fora_faixa"] = 0

    # Check for 'Sexo' column before accessing
    if "Sexo" in df.columns:
        sexo  = df["Sexo"]
        viol["sexo_fora_0_1"] = int((~sexo.between(0, 1)).sum())
    else:
        viol["sexo_fora_0_1"] = 0

    total = sum(viol.values())
    # Avoid division by zero if n is 0
    if n == 0:
        taxa_violacoes = 0.0
    else:
        taxa_violacoes = total / n

    return {
        "n_candidatos_avaliados": n,
        "contagens": viol,
        "total_violacoes": int(total),
        "taxa_violacoes": float(taxa_violacoes),
        "taxa_conformidade": float(1 - taxa_violacoes)
    }

synthetic = pd.read_excel("dados_sinteticos_realistas.xlsx")  # This synthetic DataFrame does not have 'Idade' or 'Sexo'
metricas_bruto = avaliar_regras_bruto(synthetic.copy())  # antes do pós-processamento
metricas_bruto

{'n_candidatos_avaliados': 1000,
 'contagens': {'idade_menor_18': 0,
  'idade_maior_65': 0,
  'renda_alta_para_menor': 0,
  'renda_fora_faixa': 0,
  'sexo_fora_0_1': 0},
 'total_violacoes': 0,
 'taxa_violacoes': 0.0,
 'taxa_conformidade': 1.0}